In [2]:
library(tidyverse)
library(pheatmap)
library(data.table)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

twist_barcodes <- read_csv("/mnt/dawnccle2/melange/data/guide_library/20230130_twist_library_v3.csv") %>%
  mutate(barcodeRevcomp = sapply(barcode, reverse_complement))

celltype_specific_PSI <- read_csv("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/figure_outputs/num_cell_type_celltype_specific.csv") %>% 
    separate(index_offset, into = c("index", "offset"), sep = "__", remove = F) %>% 
    separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = F) %>%
    left_join(twist_barcodes, by = c("index" = "ID")) %>%
    mutate(
      upstream_offset = as.integer(skipped_exon_start),
      downstream_offset = as.integer(skipped_exon_end),
      const_offset = as.integer(downstream_exon_start),
      upstreamIntron_len = nchar(upstreamIntronSeq),
      downstreamIntron_len = nchar(downstreamIntronSeq),
      skippedExon_len = nchar(skippedExonSeq),
      upstreamIntron_len_adj = upstreamIntron_len + upstream_offset,
      downstreamIntron_len_adj = downstreamIntron_len - downstream_offset,
      skippedExon_len_adj = skippedExon_len - upstream_offset + downstream_offset,
      upstreamIntronSeq_adj = substr(librarySequence, 1, upstreamIntron_len_adj),
      skippedExonSeq_adj = substr(librarySequence, upstreamIntron_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj),
      downstreamIntronSeq_adj = substr(librarySequence, upstreamIntron_len_adj + skippedExon_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj + downstreamIntron_len_adj)
    )

celltype_specific_altSS <- read_csv("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/figure_outputs_altSS/altSS_num_cell_type_celltype_specific.csv") %>% 
    separate(index_offset, into = c("index", "offset"), sep = "__", remove = F) %>% 
    separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = F) %>%
    left_join(twist_barcodes, by = c("index" = "ID")) %>%
    mutate(
      upstream_offset = as.integer(skipped_exon_start),
      downstream_offset = as.integer(skipped_exon_end),
      const_offset = as.integer(downstream_exon_start),
      upstreamIntron_len = nchar(upstreamIntronSeq),
      downstreamIntron_len = nchar(downstreamIntronSeq),
      skippedExon_len = nchar(skippedExonSeq),
      upstreamIntron_len_adj = upstreamIntron_len + upstream_offset,
      downstreamIntron_len_adj = downstreamIntron_len - downstream_offset,
      skippedExon_len_adj = skippedExon_len - upstream_offset + downstream_offset,
      upstreamIntronSeq_adj = substr(librarySequence, 1, upstreamIntron_len_adj),
      skippedExonSeq_adj = substr(librarySequence, upstreamIntron_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj),
      downstreamIntronSeq_adj = substr(librarySequence, upstreamIntron_len_adj + skippedExon_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj + downstreamIntron_len_adj)
    )

target_celltypes <- c("A375", "COLO783", "A172", "T47D", "MEL202", "Kelly", "HEK", "K562", "KMRC1") 

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose


Rows: 46372 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (7): ID, barcode, u

In [11]:
celltype_specific_PSI_not_mult3 <- celltype_specific_PSI %>%
      filter(skippedExon_len_adj %% 3 != 0) %>%
      filter(upsilon > 0.7 | upsilon_reverse > 0.7 | (tau > 0.75 & (row_max - row_min) > 0.4 ) | (tau_reverse > 0.75 & (row_max - row_min) > 0.4 )) %>% 
      filter((upstreamIntron_len + downstreamIntron_len + skippedExon_len) >= 210)

celltype_specific_altSS_not_mult3 <- celltype_specific_altSS %>%
      filter(skippedExon_len_adj %% 3 != 0) %>%
      filter((tau > 0.7 & (max_per_row - min_per_row) > 0.4 ) | (reverse_tau > 0.7 & (max_per_row - min_per_row) > 0.4 )) %>%
      filter((upstreamIntron_len + downstreamIntron_len + skippedExon_len) >= 210)


celltype_specific_PSI_not_mult3 %>% 
  group_by(target_cell_type) %>% 
  summarise(n = n()) %>% 
  arrange(desc(n))

celltype_specific_altSS_not_mult3 %>% 
  group_by(target_cell_type) %>% 
  summarise(n = n()) %>% 
  arrange(desc(n))


target_cell_type,n
<chr>,<int>
A375,52
COLO783,30
T47D,25
A172,24
K562,23
Kelly,19
HEK,16
HCC38,15
DAOY,6


target_cell_type,n
<chr>,<int>
A172,100
T47D,42
MEL202,13
COLO783,9
A375,8
K562,6
KMRC1,4
HEK,3
GB1,2


In [12]:
celltype_specific_PSI_not_mult3_filtered <- celltype_specific_PSI_not_mult3 %>% 
    filter(target_cell_type %in% target_celltypes) %>%
    filter(num_na_per_row < 10) %>%
    mutate(max_score = pmax(tau, tau_reverse, upsilon, upsilon_reverse, na.rm = TRUE)) %>%
    group_by(target_cell_type) %>%
    arrange(desc(max_score)) %>%
    slice_head(n = 5) %>%
    ungroup()

celltype_specific_altSS_not_mult3_filtered <- celltype_specific_altSS_not_mult3 %>% 
    filter(target_cell_type %in% target_celltypes) %>%
    filter(num_na_per_row < 10) %>%
    mutate(max_score = pmax(tau, reverse_tau, na.rm = TRUE)) %>%
    group_by(target_cell_type) %>%
    arrange(desc(max_score)) %>%
    slice_head(n = 5) %>%
    ungroup()

nrow(celltype_specific_PSI_not_mult3_filtered)
nrow(celltype_specific_altSS_not_mult3_filtered)

celltype_specific_PSI_not_mult3_filtered %>% 
    group_by(target_cell_type) %>% 
    summarise(n = n()) %>% 
    arrange(desc(n))

celltype_specific_altSS_not_mult3_filtered %>% 
    group_by(target_cell_type) %>% 
    summarise(n = n()) %>% 
    arrange(desc(n))


# Rename columns in altSS dataframe to match PSI dataframe
celltype_specific_altSS_not_mult3_filtered <- celltype_specific_altSS_not_mult3_filtered %>%
  rename(
    tau = tau,
    tau_reverse = reverse_tau,
    row_max = max_per_row,
    row_min = min_per_row,
    row_mean = mean_per_row,
    row_median = median_per_row
  ) %>%
  mutate(
    tau = NA,
    tau_reverse = NA,
  )

# Combine both dataframes
combined_filtered_events <- rbind(
  celltype_specific_PSI_not_mult3_filtered  %>% mutate(type = "PSI"),
  celltype_specific_altSS_not_mult3_filtered %>% mutate(type = "altSS")
) %>% arrange(target_cell_type, desc(max_score))
write_csv(combined_filtered_events, "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/validation/WT_validation_seq_v3.csv")

[1] 41

[1] 33

target_cell_type,n
<chr>,<int>
A172,5
A375,5
COLO783,5
HEK,5
K562,5
Kelly,5
T47D,5
MEL202,4
KMRC1,2


target_cell_type,n
<chr>,<int>
A172,5
A375,5
COLO783,5
K562,5
MEL202,5
T47D,5
HEK,2
KMRC1,1


In [3]:
# Save another set that's all that's high in Kelly.

target_celltypes <- c("Kelly") 

celltype_specific_PSI_not_mult3 <- celltype_specific_PSI %>%
      filter(skippedExon_len_adj %% 3 != 0) %>%
      filter(upsilon > 0.7 | upsilon_reverse > 0.7 | (tau > 0.75 & (row_max - row_min) > 0.4 ) | (tau_reverse > 0.75 & (row_max - row_min) > 0.4 )) %>% 
      filter((upstreamIntron_len + downstreamIntron_len + skippedExon_len) >= 210)

celltype_specific_altSS_not_mult3 <- celltype_specific_altSS %>%
      filter(skippedExon_len_adj %% 3 != 0) %>%
      filter((tau > 0.7 & (max_per_row - min_per_row) > 0.4 ) | (reverse_tau > 0.7 & (max_per_row - min_per_row) > 0.4 )) %>%
      filter((upstreamIntron_len + downstreamIntron_len + skippedExon_len) >= 210)


celltype_specific_PSI_not_mult3 %>% 
  group_by(target_cell_type) %>% 
  summarise(n = n()) %>% 
  arrange(desc(n))

celltype_specific_altSS_not_mult3 %>% 
  group_by(target_cell_type) %>% 
  summarise(n = n()) %>% 
  arrange(desc(n))

celltype_specific_PSI_not_mult3_filtered <- celltype_specific_PSI_not_mult3 %>% 
    filter(target_cell_type %in% target_celltypes) %>%
    filter(num_na_per_row < 10) %>%
    mutate(max_score = pmax(tau, tau_reverse, upsilon, upsilon_reverse, na.rm = TRUE)) %>%
    group_by(target_cell_type) %>%
    arrange(desc(max_score)) %>%
    ungroup()

celltype_specific_altSS_not_mult3_filtered <- celltype_specific_altSS_not_mult3 %>% 
    filter(target_cell_type %in% target_celltypes) %>%
    filter(num_na_per_row < 10) %>%
    mutate(max_score = pmax(tau, reverse_tau, na.rm = TRUE)) %>%
    group_by(target_cell_type) %>%
    arrange(desc(max_score)) %>%
    ungroup()

nrow(celltype_specific_PSI_not_mult3_filtered)
nrow(celltype_specific_altSS_not_mult3_filtered)

celltype_specific_PSI_not_mult3_filtered %>% 
    group_by(target_cell_type) %>% 
    summarise(n = n()) %>% 
    arrange(desc(n))

celltype_specific_altSS_not_mult3_filtered %>% 
    group_by(target_cell_type) %>% 
    summarise(n = n()) %>% 
    arrange(desc(n))


# Rename columns in altSS dataframe to match PSI dataframe
celltype_specific_altSS_not_mult3_filtered <- celltype_specific_altSS_not_mult3_filtered %>%
  rename(
    tau = tau,
    tau_reverse = reverse_tau,
    row_max = max_per_row,
    row_min = min_per_row,
    row_mean = mean_per_row,
    row_median = median_per_row
  ) %>%
  mutate(
    tau = NA,
    tau_reverse = NA,
  )

# Combine both dataframes
combined_filtered_events <- rbind(
  celltype_specific_PSI_not_mult3_filtered  %>% mutate(type = "PSI"),
  celltype_specific_altSS_not_mult3_filtered %>% mutate(type = "altSS")
) %>% arrange(target_cell_type, desc(max_score))
write_csv(combined_filtered_events, "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/validation/WT_validation_seq_kelly_only.csv")

target_cell_type,n
<chr>,<int>
A375,52
COLO783,30
T47D,25
A172,24
K562,23
Kelly,19
HEK,16
HCC38,15
DAOY,6


target_cell_type,n
<chr>,<int>
A172,100
T47D,42
MEL202,13
COLO783,9
A375,8
K562,6
KMRC1,4
HEK,3
GB1,2


[1] 19

[1] 0

target_cell_type,n
<chr>,<int>
Kelly,19


target_cell_type,n
<chr>,<int>
